# 📊 100-Day Data Science Challenge
## Day 7: Tuples, Mutability & Memory References

Welcome to Day 7! Today, we delve into how Python handles data structure mutability, memory allocation, and copy mechanisms. Understanding these core concepts is critical for debugging complex data pipelines and avoiding unexpected bugs in data science.

### Today's Topics:
1. **Tuples Basics**
2. **Packing & Unpacking**
3. **Mutability vs Immutability**
4. **Memory References & Identity**
5. **Shallow vs Deep Copy**
6. **Pass-by-Object-Reference**
7. **Practical Exercise: Troubleshooting Config State bugs**

--- 
### 1. Tuples Basics
A tuple is an ordered, immutable collection of elements. Tuples are written with round brackets `( )` (though parentheses are technically optional in many cases).

In [ ]:
# Tuple Creation
tup1 = (1, 2, 3)
tup2 = "a", "b", "c"  # Without parentheses
print(f"tup1: {tup1}, type: {type(tup1)}")
print(f"tup2: {tup2}, type: {type(tup2)}")

# WARNING: Single element tuple requires a trailing comma
not_tuple = (5)
is_tuple = (5,)
print(f"(5) type: {type(not_tuple)}")
print(f"(5,) type: {type(is_tuple)}")

# Indexing and Slicing
print(f"First item: {tup1[0]}")
print(f"Slice of tup2: {tup2[1:]}")

# Attempting modification raises TypeError
try:
    tup1[0] = 99
except TypeError as e:
    print(f"Error caught: {e}")

--- 
### 2. Packing & Unpacking
Python allows packing values into a tuple and unpacking tuples directly into variables. We can also swap variables quickly and use extended unpacking with the `*` operator.

In [ ]:
# Packing
packed = 10, "data", 3.14
print("Packed:", packed)

# Unpacking
val1, val2, val3 = packed
print(f"Unpacked: {val1}, {val2}, {val3}")

# Swapping variables
a, b = 1, 99
print(f"Before swap: a={a}, b={b}")
a, b = b, a
print(f"After swap:  a={a}, b={b}")

# Extended unpacking (*)
numbers = (10, 20, 30, 40, 50)
first, *middle, last = numbers
print(f"first: {first}, middle: {middle}, last: {last}")

--- 
### 3. Mutability vs Immutability & Identity
- **Immutable Objects**: cannot be modified after creation (e.g., int, float, str, tuple, bool).
- **Mutable Objects**: can be modified in place (e.g., list, dict, set).
- **Equality (`==`)**: Checks if two variables have the same values.
- **Identity (`is`)**: Checks if two variables point to the exact same object in memory (`id(a) == id(b)`).

In [ ]:
# Comparing Equality vs Identity
list1 = [1, 2, 3]
list2 = [1, 2, 3]
list3 = list1

print(f"list1 == list2: {list1 == list2} (Same values)")
print(f"list1 is list2: {list1 is list2} (Same object reference?)")
print(f"list1 is list3: {list1 is list3} (Same object reference?)")

# Mutating shared references affects all labels
list3.append(4)
print(f"list1 after modifying list3: {list1}")

--- 
### 4. Small Integer Interning & Referential Immutability
- **Integer Interning**: Python caches small integers in the range `[-5, 256]` to save memory and execution time.
- **Referential Immutability**: If an immutable object (like a tuple) contains a reference to a mutable object (like a list), the tuple itself cannot be reassigned, but the inner mutable object can still be altered.

In [ ]:
# Integer Interning
x = 100
y = 100
print(f"100 is cached? x is y: {x is y}")

a = 300
b = 300
print(f"300 is cached? a is b: {a is b} (Depends on execution context)")

# Tuple with Mutable Element
shared_lst = [10, 20]
t = (1, shared_lst, "Python")
print("Initial tuple:", t)

try:
    t[1] = [30, 40]  # Throws TypeError because the reference at index 1 is locked.
except TypeError as e:
    print("TypeError:", e)

# Modifying mutable list in place
t[1].append(30)
print("Modified tuple:", t)

--- 
### 5. Shallow Copy vs Deep Copy
- **Shallow Copy (`copy.copy`)**: Creates a new collection, but populates it with references to the original nested items. Modifying nested lists will affect the original.
- **Deep Copy (`copy.deepcopy`)**: Creates a new collection and recursively creates copies of all nested objects. Changes to nested items will NOT affect the original.

In [ ]:
import copy

original_list = [[1, 2], [3, 4]]

# Creating copies
shallow = copy.copy(original_list)
deep = copy.deepcopy(original_list)

# Modify nested list in shallow copy
shallow[0].append(99)
# Modify nested list in deep copy
deep[1].append(999)

print("Original:    ", original_list)
print("Shallow Copy:", shallow)
print("Deep Copy:   ", deep)

--- 
### 6. Pass-by-Object-Reference
Python uses pass-by-object-reference (or call-by-sharing):
- Rebinding a variable within a function changes the local name binding but doesn't affect the caller.
- Modifying a mutable object in-place inside a function WILL affect the caller's object.

In [ ]:
def rebind_and_mutate(num, lst):
    num = 500  # Rebinding local variable (immutable)
    lst.append(99)  # Mutating list in place (mutable)
    lst = [7, 8, 9]  # Rebinding local variable (lst now points to new list)

n = 10
l = [1, 2]
print(f"Before: n={n}, l={l}")
rebind_and_mutate(n, l)
print(f"After:  n={n}, l={l}")

--- 
### 7. Practical Exercise: Solving a Shared State Config Bug
Below, we simulate a common bug where multiple manager instances share the same default nested configuration because they reference the same dictionary in memory, and show how using `copy.deepcopy` resolves this issue.

In [ ]:
class BadConfigManager:
    def __init__(self, default_config):
        # Buggy: shares references to nested dicts
        self.config = default_config
        
    def set_db_host(self, host):
        self.config["database"]["host"] = host

class GoodConfigManager:
    def __init__(self, default_config):
        # Safe: creates a completely independent copy
        self.config = copy.deepcopy(default_config)
        
    def set_db_host(self, host):
        self.config["database"]["host"] = host

# Shared default settings
default_settings = {
    "app": "TestApp",
    "database": {"host": "localhost", "port": 5432}
}

print("--- Testing Buggy Config Manager ---")
dev_config = BadConfigManager(default_settings)
prod_config = BadConfigManager(default_settings)

print("Updating dev DB host to 'dev-db'...")
dev_config.set_db_host("dev-db")
print(f"prod DB host: {prod_config.config['database']['host']} (Bug: prod changed to dev!)")

# Reset default settings
default_settings["database"]["host"] = "localhost"

print("\n--- Testing Safe Config Manager ---")
dev_config_safe = GoodConfigManager(default_settings)
prod_config_safe = GoodConfigManager(default_settings)

print("Updating dev DB host to 'dev-db'...")
dev_config_safe.set_db_host("dev-db")
print(f"prod DB host: {prod_config_safe.config['database']['host']} (Success: prod is unaffected!)")
print(f"default settings host: {default_settings['database']['host']}")